In [1]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import random
import math

import pandas as pd


import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots

In [3]:
import sys
from pathlib import Path


In [4]:
from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

# Helper Functions

In [5]:
#from splitflow.unifying_algo_analysis_helper import *
from splitflow.set_zorder_functions import *

Using device: cpu


In [6]:
def clean_labels_course_ylim(ax, show_labels, spines=['left', 'bottom']):
    #set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
    #set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)
    ax.set_rasterization_zorder(0)

    ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_ylim(-np.pi, np.pi)
    ax.set_xlim(-.2, 5)
    ax.set_xticks([-0.2, 0, 0.5, 1, 2, 3, 4, 5])

    if 0:
        ax.set_xticklabels(['', '0', '.5', '', '2', '3', '4', '5'])
    else:
        ax.set_xticklabels([])
    
    if show_labels:
        ax.set_yticklabels(['$-\pi$', '','$0$','','$\pi$',])
    else:
        ax.set_yticklabels([])
        
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])
    
    if show_labels:
        ax.set_ylabel('Upwind') #'Course direction', labelpad=-2)
        ax.set_xlabel('') #Time relative to flash (s)', labelpad=1)
    else:
        ax.set_ylabel('')
        ax.set_xlabel('')
    
    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)
    
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [7]:
def clean_labels(ax, show_labels, spines=['left', 'bottom']):
    #set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
    #set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)
    ax.set_rasterization_zorder(0)

    ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_ylim(-np.pi, np.pi)
    ax.set_xlim(-.2, 5)
    ax.set_xticks([-0.2, 0, 0.5, 1, 2, 3, 4, 5])

    if show_labels:
        ax.set_xticklabels(['', '0', '.5', '', '2', '3', '4', '5'])
    else:
        ax.set_xticklabels([])
    
    if show_labels:
        ax.set_yticklabels(['$-\pi$', '','$0$','','$\pi$',])
    else:
        ax.set_yticklabels([])
        
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])
    
    if show_labels:
        ax.set_ylabel('Course direction', labelpad=-2)
        ax.set_xlabel('Time relative to flash (s)', labelpad=1)
    else:
        ax.set_ylabel('')
        ax.set_xlabel('')
    
    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)
    
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [8]:
def clean_labels_hist(ax, show_labels, spines=['left', 'bottom']):
    #set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
    #set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)
    ax.set_rasterization_zorder(0)

    ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_xlim(-np.pi, np.pi)
    ax.set_ylim(0, 1.5)
    ax.set_yticks([0, 0.5, 1, 1.5])

    if show_labels:
        pass #ax.set_xticklabels(['', '0', '.68', '', '2', '3', '4', '5'])
    else:
        ax.set_yticklabels([])
    
    if show_labels:
        ax.set_xticklabels(['$-\pi$', '','$0$','','$\pi$',])
    else:
        ax.set_xticklabels([])
        
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])
    
    if show_labels:
        ax.set_xlabel('Course direction during surge', labelpad=1)
        ax.set_ylabel('Density', labelpad=0)
    else:
        ax.set_ylabel('')
        ax.set_xlabel('')
    
    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)
    
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [9]:
FIGURE_NAME = 'supplemental_unifying_analysis_variable_wind_surge.svg'

In [10]:
TRANSLATION = True

In [11]:
COURSE_MARKER_SIZE = 2
COURSE_ALPHA_MULTIPLIER = 3

In [12]:
class LabelToMetadata:
    def __init__(self):
        self.flash = metadata = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/' + 'varwind_N700_flash_12.0_translationFalse' + '.parquet': [1, 'gray', '12', 0], 
                                    str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/' + 'varwind_N700_flash_20.0_translationFalse' + '.parquet': [2, 'gray', '20', 0.03],
                                    str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/' + 'varwind_N700_flash_23.0_translationFalse' + '.parquet': [3, 'gray', '23', 0.1],
                                    str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/' + 'varwind_N700_flash_28.0_translationFalse' + '.parquet': [4, '#a245ffff', '28', 0.2],
                                    str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/' + 'varwind_N700_flash_40.0_translationFalse' + '.parquet': [5, 'gray', '40', 0.3],
                                    str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/' + 'varwind_N700_flash_90.0_translationFalse' + '.parquet': [6, 'gray', '90', 0.6],
                                   }
        self.sham = metadata = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-12-fan_0-cms_all-traj_align_SHAM' + '.parquet': [1, 'gray', '12', 0], 
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-20-fan_0-03-cms_all-traj_align_SHAM' + '.parquet': [2, 'gray', '20', 0.03],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-23-fan_0-1-cms_all-traj_align_SHAM' + '.parquet': [3, '#bd7bffff', '23', 0.1],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-28-fan_0-2-cms_all-traj_align_SHAM' + '.parquet': [4, 'gray', '28', 0.2],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-40-fan_0-3-cms_all-traj_align_SHAM' + '.parquet': [5, 'gray', '40', 0.3],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-90-fan_0-6-cms_all-traj_align_SHAM' + '.parquet': [6, 'gray', '90', 0.6],
                               }

In [13]:
def get_filename_for_wind_type(metadata, windtype):
    filename = None
    for key, val in metadata.items():
        if windtype in val:
            filename = key
    return filename

In [14]:
def get_trajec_filename_from_unifying_filename(unifying_filename=None):
    return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet'


In [15]:
def get_filenames_for_metadata_windtype(metadata, windtype):
    unifying_filename = get_filename_for_wind_type(metadata, windtype)
    trajectory_filename = None
    df = None
    unifying_algo_data = None
    
    if unifying_filename != 'None':
        unifying_algo_data = pd.read_parquet(unifying_filename)
    
        trajectory_filename = get_trajec_filename_from_unifying_filename(unifying_filename)
        if '.hdf' in trajectory_filename:
            df = pd.read_hdf(trajectory_filename)
        else:
            df = pd.read_parquet(trajectory_filename)
    
    else:
        unifying_algo_data = None

    print(unifying_filename)
    print(trajectory_filename)
    return unifying_algo_data, df

In [16]:
def get_obj_ids_above_threshold(df, threshold, threshold_operation='>', time_col='time_relative_to_flash', 
                                 id_col='obj_id_unique_event', 
                                 course_col='course_smoothish'):
    pre_flash = df[df[time_col] < 0]
    mean_abs_course = pre_flash.groupby(id_col)[course_col].apply(lambda x: x.abs().mean())

    if threshold_operation == '>':
        return mean_abs_course[mean_abs_course > threshold].index.tolist()
    elif threshold_operation == '<':
        return mean_abs_course[mean_abs_course < threshold].index.tolist()

In [17]:
label = 'sham'
label_to_metadata = LabelToMetadata()
fifi_figure_label = 'unifying_' + label
metadata = label_to_metadata.__getattribute__(label)

# Make course and surge plots

repeat for labels:

'flash', 'sham'

In [18]:
flash_or_sham = 'sham'

In [19]:
df = pd.read_parquet(get_trajec_filename_from_unifying_filename())

In [20]:
def plot_course_filtered_and_surge(df, wind_speed, flash_or_sham, 
                                   layout=None, 
                                   show_labels=False,
                                   reference_hist=None,
                                   reference_bins=None):
    if layout is None:
        fig = plt.figure(figsize=(3,7))
        ax_course = fig.add_subplot(311)
        ax_filtered = fig.add_subplot(312)
        ax_surge = fig.add_subplot(313)
    else:
        ax_course = layout.axes[('course_' + flash_or_sham, str(wind_speed))]
        ax_filtered = layout.axes[('course_filtered_' + flash_or_sham, str(wind_speed))]
        ax_surge = layout.axes[('course_surge_' + flash_or_sham, str(wind_speed))]

    if flash_or_sham == 'flash':
        intensity = 255.
    else:
        intensity = 0


    if flash_or_sham == 'flash':
        opto_shading = 'red'
    else:
        opto_shading = 'gray'
        
    braid_df = df[(df.fan_speed_percent==wind_speed) & (df.intensity==intensity)]
    
    ax_course.fill_betweenx([-np.pi, np.pi], 0, 0.500, edgecolor='none', facecolor=opto_shading, alpha=0.3)
    braid_analysis_plots.plot_column_vs_time(braid_df,
                                                column='course_smoothish',
                                                time_key='time_relative_to_flash',
                                                norm_columns_to_min_max=True,
                                                norm_columns_to_sum=False,
                                                norm_columns_to_min_max_smoothing=20,
                                                cmap='bone_r',
                                                vmin=0,
                                                vmax=1,
                                                bin_y=None,
                                                bin_x=None,
                                                res_y=0.05,
                                                res_x=0.02,
                                                ax=ax_course,
                                                interpolation='nearest',
                                                return_array=False,
                                            )
    clean_labels_course_ylim(ax_course, show_labels)

    ## Filtered
    obj_id_not_upwind = get_obj_ids_above_threshold(braid_df, np.pi/3, '>')
    braid_df_not_upwind = braid_df[braid_df.obj_id_unique_event.isin(obj_id_not_upwind)]

    
    ax_filtered.fill_betweenx([-np.pi, np.pi], 0, 0.500, edgecolor='none', facecolor=opto_shading, alpha=0.3)
    braid_analysis_plots.plot_column_vs_time(braid_df_not_upwind,
                                                column='course_smoothish',
                                                time_key='time_relative_to_flash',
                                                norm_columns_to_min_max=True,
                                                norm_columns_to_sum=False,
                                                norm_columns_to_min_max_smoothing=20,
                                                cmap='bone_r',
                                                vmin=0,
                                                vmax=1,
                                                bin_y=None,
                                                bin_x=None,
                                                res_y=0.05,
                                                res_x=0.02,
                                                ax=ax_filtered,
                                                interpolation='nearest',
                                                return_array=False,
                                            )
    
    ax_filtered.fill_betweenx([-np.pi, np.pi], 0.8, 1.0, edgecolor='dodgerblue', facecolor='none', alpha=1)
    clean_labels(ax_filtered, show_labels)
    
    surge = braid_df_not_upwind.query('time_relative_to_flash>0.8 & time_relative_to_flash<1.')
    hist, bins, patches = ax_surge.hist(surge.course_smoothish.values, bins=np.linspace(-np.pi, np.pi, 20), color='dodgerblue', density=True)

    if reference_hist is not None:
        ax_surge.stairs(reference_hist, reference_bins, color='black')
    if wind_speed == 12:
        ax_surge.stairs(hist, bins, color='black')

    clean_labels_hist(ax_surge, show_labels)

    item = layout.svgitems['text_n_' + str(wind_speed) + '_' + flash_or_sham]
    N_trajecs = len(braid_df.obj_id_unique_event.unique())
    item.text = 'n=' + str(N_trajecs)
        
    return hist, bins

In [21]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [22]:
for wind_speed in [12, 20, 23, 28, 40, 90]:
    print(wind_speed)
    if wind_speed == 12:
        hist, bins = plot_course_filtered_and_surge(df, wind_speed, flash_or_sham, layout=layout, show_labels=True)
    else:
        _, _ = plot_course_filtered_and_surge(df, wind_speed, flash_or_sham, layout=layout, show_labels=False,
                                      reference_hist=hist, reference_bins=bins)

12


20
23


28
40


90


In [23]:
layout.apply_svg_attrs()
layout.append_figure_to_layer(layout.figures['course_'+flash_or_sham], 'course_'+flash_or_sham, cleartarget=True)
layout.append_figure_to_layer(layout.figures['course_filtered_'+flash_or_sham], 'course_filtered_'+flash_or_sham, cleartarget=True)
layout.append_figure_to_layer(layout.figures['course_surge_'+flash_or_sham], 'course_surge_'+flash_or_sham, cleartarget=True)
layout.write_svg(FIGURE_NAME)